In [1]:
# === Librerías == #
# Importamos las clases que se requieren para manejar los agentes (Agent)
# y su entorno (Model). Cada modelo puede contener múltiples agentes.
from mesa import Agent, Model
from mesa.space import MultiGrid

# Para este problema, usaremos un espacio continuo.
from mesa.space import ContinuousSpace

# Haremos uso de ''DataCollector'' para obtener información de cada paso
# de la simulación.
from mesa.datacollection import DataCollector

import random
from enum import Enum

# matplotlib lo usaremos crear una animación de cada uno de los pasos
# del modelo.
%matplotlib inline
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.animation as animation
plt.rcParams["animation.html"] = "jshtml"
matplotlib.rcParams['animation.embed_limit'] = 2**128

# Importamos los siguientes paquetes para el mejor manejo de valores
# numéricos.
import numpy as np
import pandas as pd

In [2]:
# ==== Modelo === #

# ==== Enums de Estado ==== #
class EstadoFuego(Enum):
    LIMPIO = 0
    HUMO = 1
    FUEGO = 2

class TipoPOI(Enum):
    VICTIMA = 1
    FALSA_ALARMA = 2

# ==== Objetos de Juego ==== #
class POI():
    def __init__(self, tipo):
        self.tipo = tipo # TipoPOI (Victima o Falsa Alarma)
        self.revelado = False # Comienza boca abajo

# ==== Clases de modelo ==== #
class Nodo():
    def __init__(self, pos):
        self.pos = pos # Vector2D de posicion (x,y)
        self.estado_fuego = EstadoFuego.LIMPIO # Estado del hazard
        self.contenido = [] # Lista para almacenar Firefighters, POIs, etc.
        self.vecinos = {} # Diccionario de vecinos {Nodo: Arista}





In [3]:

class TipoArista(Enum):
    MURO = 1
    PUERTA = 2

class Arista():
    def __init__(self, tipo):
        self.tipo = tipo 

class Puerta(Arista):
    def __init__(self):
        super().__init__(TipoArista.PUERTA)
        self.cerrado = True

    def abrir(self):
        self.cerrado = False
        
    def destruir(self):
        self.cerrado = False # En el juego, una puerta destruida cuenta como espacio abierto

class Muro(Arista):
    def __init__(self):
        super().__init__(TipoArista.MURO)
        self.hp = 2

    def golpear(self):
        if self.hp > 0:
            self.hp -= 1
    

In [4]:
class FlashPointModel(Model):
    def __init__(self, numAgents, width, height):
        super().__init__()
        self.grid = MultiGrid(width, height, torus=False) # Grid de vectores2D
        self.bolsa_poi = [] #Bolsa de Objetivos

        ## Data collector (despues lo hacemos) ##

        ## === Crear matriz de Nodos y llenar la grid de Mesa ==

        # 1. Crear matriz de Nodos
        self.mapa_nodos = {} 
        for x in range(width):
            for y in range(height):
                nodo = Nodo(pos=(x, y))
                self.mapa_nodos[(x, y)] = nodo

        # 2. Inicializamos relaciones ortogonales
        self._conectar_vecinos_base(width, height)

        # 3. Colocar Muros y Puertas 
        self._cargar_infraestructura_tablero()
        
        # 4. Preparar Setup Familiar
        self._preparar_juego_familiar()

    def _conectar_vecinos_base(self, width, height):
        for (x, y), nodo in self.mapa_nodos.items():
            direcciones = [(x+1, y), (x-1, y), (x, y+1), (x, y-1)]
            for nx, ny in direcciones:
                if 0 <= nx < width and 0 <= ny < height:
                    nodo_vecino = self.mapa_nodos[(nx, ny)]
                    nodo.vecinos[nodo_vecino] = None 

    def _colocar_borde(self, pos_a, pos_b, objeto_arista):
        nodo_a = self.mapa_nodos[pos_a]
        nodo_b = self.mapa_nodos[pos_b]
        nodo_a.vecinos[nodo_b] = objeto_arista
        nodo_b.vecinos[nodo_a] = objeto_arista

    def _cargar_infraestructura_tablero(self):
        # --- PUERTAS (8 Puertas Totales) ---
        lista_muros = [
            # --- EXTERIORES ---
            # Perímetro Izquierdo
            ((0,1), (1,1)), ((0,2), (1,2)), ((0,3), (1,3)), ((0,5), (1,5)), ((0,6), (1,6)),
            # Perímetro Inferior
            ((1,0), (1,1)), ((2,0), (2,1)),
            ((4,0), (4,1)), ((5,0), (5,1)), ((6,0), (6,1)), ((7,0), (7,1)), ((8,0), (8,1)),
            # Perímetro Superior (Se agregó el muro en x=4, se quitó en x=6)
            ((1,6), (1,7)), ((2,6), (2,7)), ((3,6), (3,7)), ((4,6), (4,7)), ((5,6), (5,7)), ((7,6), (7,7)), ((8,6), (8,7)),
            # Perímetro Derecho
            ((8,1), (9,1)), ((8,2), (9,2)), ((8,4), (9,4)), ((8,5), (9,5)), ((8,6), (9,6)),

            # --- INTERIORES VERTICALES (Separan X y X+1) ---
             
            ((5,2), (6,2)), ((7,2), (8,2)),
            ((2,3), (3,3)), 
            ((6,4), (7,4)),
            ((3,5), (4,5)),
            ((5,6), (6,6)), # <-- El muro faltante arriba de la puerta 5,5-6,5

            # --- INTERIORES HORIZONTALES (Separan Y y Y+1) ---
            ((1,2), (1,3)), ((2,2), (2,3)), ((3,2), (3,3)), # <-- Muros agregados entre fila 2 y 3
            ((5,2), (5,3)),
            ((6,2), (6,3)), ((7,2), (7,3)), ((8,2), (8,3)),
            # Muro continuo entre fila 4 y 5
            ((3,4), (3,5)), ((4,4), (4,5)), ((5,4), (5,5)), ((6,4), (6,5)), ((7,4), (7,5))
        ]
        
        lista_puertas = [
            # --- EXTERIORES ---
            ((3,0), (3,1)), # Abajo
            ((0,4), (1,4)), # Izquierda
            ((6,6), (6,7)), # Arriba (Movida desde x=4)
            ((8,3), (9,3)), # Derecha
            
            # --- INTERIORES ---
            ((7,1), (8,1)), # Vertical
            ((5,1), (6,1)), # Vertical
            ((4,2), (4,3)), # Horizontal
            ((6,3), (7,3)), # Vertical
            ((2,4), (3,4)), # Vertical
            ((8,4), (8,5)), # Horizontal
            ((3,6), (4,6)), # Vertical
            ((5,5), (6,5))  # Vertical
        ]

        for pos_a, pos_b in lista_muros:
            if pos_a in self.mapa_nodos and pos_b in self.mapa_nodos:
                self._colocar_borde(pos_a, pos_b, Muro())

        for pos_a, pos_b in lista_puertas:
             if pos_a in self.mapa_nodos and pos_b in self.mapa_nodos:
                self._colocar_borde(pos_a, pos_b, Puerta())

    def _preparar_juego_familiar(self):
        fuegos_iniciales = [(2,2), (2,3), (3,2), (3,4), (4,4), (5,5), (6,5), (7,6), (4,2), (5,3)]
        for pos in fuegos_iniciales:
            if pos in self.mapa_nodos:
                self.mapa_nodos[pos].estado_fuego = EstadoFuego.FUEGO
                
        self.bolsa_poi = [TipoPOI.VICTIMA] * 10 + [TipoPOI.FALSA_ALARMA] * 5
        random.shuffle(self.bolsa_poi)
        
        pois_iniciales = [(2,4), (5,1), (7,4)]
        for pos in pois_iniciales:
            if pos in self.mapa_nodos:
                tipo_poi = self.bolsa_poi.pop()
                self.mapa_nodos[pos].contenido.append(POI(tipo_poi))


## === Visualizacion DEBUG === ##
    def imprimir_tablero_debug(self):
        print("\n" + "=" * 48)
        print("             DEBUG: MAPA DE FLASH POINT")
        print("    [ ]=Limpio [H]=Humo [F]=Fuego | ? = POI Oculto")
        print("    ║/═ Muro (2HP) | │/─ Muro (1HP) | D/d Puerta")
        print("=" * 48 + "\n")

        # Dibujar la cuadrícula de arriba hacia abajo
        for y in range(self.grid.height - 1, -1, -1):
            
            # 1. Renglón superior del nodo (Muros/Puertas Horizontales)
            linea_horizontal = "  "
            for x in range(self.grid.width):
                nodo_actual = self.mapa_nodos[(x, y)]
                nodo_arriba = self.mapa_nodos.get((x, y + 1))
                
                # Borde superior
                if nodo_arriba and nodo_arriba in nodo_actual.vecinos:
                    borde = nodo_actual.vecinos[nodo_arriba]
                    # Solo agrega el bloque de 5 caracteres exactos
                    linea_horizontal += self._simbolo_borde_horizontal(borde) 
                else:
                    linea_horizontal += "─────"  # Borde exterior de 5 caracteres
            print(linea_horizontal)

            # 2. Renglón del nodo (Nodos y Muros/Puertas Verticales)
            linea_nodos = f"{y} "
            for x in range(self.grid.width):
                nodo_actual = self.mapa_nodos[(x, y)]
                nodo_derecha = self.mapa_nodos.get((x + 1, y))

                # Visualizar estado del fuego (F, H, o Limpio)
                simbolo_fuego = " "
                if nodo_actual.estado_fuego == EstadoFuego.FUEGO: simbolo_fuego = "F"
                elif nodo_actual.estado_fuego == EstadoFuego.HUMO: simbolo_fuego = "H"
                
                # Visualizar si hay un POI
                simbolo_poi = "?" if any(isinstance(c, POI) for c in nodo_actual.contenido) else " "
                
                # Esto compone los primeros 4 caracteres: ej. "[F?]" o "[  ]"
                linea_nodos += f"[{simbolo_fuego}{simbolo_poi}]"

                # Borde derecho (El 5to caracter)
                if nodo_derecha and nodo_derecha in nodo_actual.vecinos:
                    borde = nodo_actual.vecinos[nodo_derecha]
                    linea_nodos += self._simbolo_borde_vertical(borde)
                else:
                    linea_nodos += " "  # Espacio libre (1 caracter)
            print(linea_nodos)

        # Renglón inferior final y ejes X centrado
        print("  " + "─────" * self.grid.width)
        
        # Centra los números del eje X bajo los bloques de 5 caracteres
        eje_x = "  " + "".join(f"  {x}  " for x in range(self.grid.width))
        print(eje_x + "\n")

    def _simbolo_borde_horizontal(self, borde):
        # Devuelve SIEMPRE strings de exactamente 5 caracteres
        if borde is None: return "     " 
        if isinstance(borde, Muro):
            if borde.hp == 2: return "════ "
            elif borde.hp == 1: return "──── "
            return "     " 
        if isinstance(borde, Puerta):
            return "  D  " if borde.cerrado else "  d  "
        return "     "

    def _simbolo_borde_vertical(self, borde):
        # Devuelve SIEMPRE strings de exactamente 1 caracter
        if borde is None: return " "
        if isinstance(borde, Muro):
            if borde.hp == 2: return "║"
            elif borde.hp == 1: return "│"
            return " "  
        if isinstance(borde, Puerta):
            return "D" if borde.cerrado else "d"
        return " "



# ==== Sistema de DTO Python -> Unity === #

#Esta funcion aplana el mapa de nodos en una estructura facil de leer poder el serializador de JSON y Unity
#Esta es la estructura esperada: 
# "nodes": [
#{"x": 0, "y": 0, "fuego": "SIN_FUEGO", "poi": null} ]
def _exportar_nodos_dto(self):
    nodos_lista = []
    
    # Iteramos sobre los valores del diccionario mapa_nodos
    for nodo in self.mapa_nodos.values():
        x, y = nodo.pos  # Desempaquetamos el vector 2D (x, y)
        
        # 1. Buscar si hay algún POI en la lista de contenido
        poi_dto = None
        for item in nodo.contenido:
            if isinstance(item, POI):
                poi_dto = {
                    "tipo": item.tipo.name,       # "VICTIMA" o "FALSA_ALARMA"
                    "revelado": item.revelado     # True o False
                }
                break  # Tomamos el primer POI encontrado
        
        # 2. Construir el diccionario plano del Nodo
        nodo_dto = {
            "x": x,
            "y": y,
            "fuego": nodo.estado_fuego.name,  # "LIMPIO", "HUMO" o "FUEGO"
            "poi": poi_dto
        }
        
        nodos_lista.append(nodo_dto)
        
    return nodos_lista

#Esta funcion itera sobre todos los nodos sacando los Objetos arista dentro de sus diccionarios de adyacencia
# Si el nodo adyacente tiene un objeto arista valido, obtiene sus datos y lo GUARDA en una lista de procesados.
#De esta manera no procesamos 2 veces el mismo objeto
def _exportar_aristas_dto(self):
    aristas_lista = []
    procesados = set()  # Para guardar referencias de objetos Arista ya exportados
    
    for nodo in self.mapa_nodos.values():
        pos_a = nodo.pos  # Vector2D (x, y) del nodo origen
        
        for nodo_vecino, arista in nodo.vecinos.items():
            # Si hay un objeto Arista y no lo hemos procesado antes
            if arista is not None and arista not in procesados:
                procesados.add(arista)  # Marcamos este objeto Arista como visitado
                pos_b = nodo_vecino.pos  # Tupla (x, y) del nodo vecino
                
                # 1. Base del DTO con coordenadas primitivas
                arista_dto = {
                    "posA": {"x": pos_a[0], "y": pos_a[1]},
                    "posB": {"x": pos_b[0], "y": pos_b[1]},
                    "tipo": arista.tipo.name  # "MURO" o "PUERTA"
                }
                
                # 2. Extraer atributos específicos según la subclase
                if isinstance(arista, Puerta):
                    arista_dto["cerrado"] = arista.cerrado
                elif isinstance(arista, Muro):
                    arista_dto["hp"] = arista.hp
                
                aristas_lista.append(arista_dto)
                
    return aristas_lista

# === funcion general para obtener el DTO completo de la fase de Setup === #
def get_setup_dto(self):
    return {
        "width": self.grid.width,
        "height": self.grid.height,
        "nodes": self._exportar_nodos_dto(),
        "edges": self._exportar_aristas_dto()
    }


In [5]:
modelo = FlashPointModel(numAgents=0, width=10, height=8)
#modelo.imprimir_tablero_debug()


In [6]:
## Configuracion de servidor y conexion con Unity ## 

In [7]:
%pip install flask flask-cors

Note: you may need to restart the kernel to use updated packages.


In [8]:
import threading
from flask import Flask, jsonify, request
from flask_cors import CORS

# 1. Creación de la app web
app = Flask(__name__) # Le dice a Flask cual es la "Celda Principal" 
CORS(app) # Habilitar llamadas desde cualquier origen (incluyendo Unity)

# 2. Definición de la Ruta / Endpoint
#Este es un Decorador, la funcion que este despues de este es la que se 
#va a ejecutar cuando se llame a esta ruta. NO PONER COMENTARIOS ENTRE EL DECOR Y EL MIDDLEWARE.
@app.route('/api/process', methods=['GET']) 
def GetSetupData():
    return jsonify(modelo.get_setup_dto()), 200


# 3. Estrategia de Ejecución en Hilos (Thread) para Jupyter Notebook
def run_server():
    # Runs backend on port 5000
    app.run(host='127.0.0.1', port=5000, debug=False, use_reloader=False)

# Levantamos el servidor en un hilo secundario
server_thread = threading.Thread(target=run_server)
server_thread.daemon = True
server_thread.start()

print("🚀 Servidor escuchando en http://127.0.0.1:5000/api/process")

🚀 Servidor escuchando en http://127.0.0.1:5000/api/process
 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
[2026-09-01 14:29:17,968] ERROR in app: Exception on /api/process [GET]
Traceback (most recent call last):
  File "/opt/miniconda3/envs/multiagentes/lib/python3.12/site-packages/flask/app.py", line 1511, in wsgi_app
    response = self.full_dispatch_request()
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/miniconda3/envs/multiagentes/lib/python3.12/site-packages/flask/app.py", line 919, in full_dispatch_request
    rv = self.handle_user_exception(e)
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/miniconda3/envs/multiagentes/lib/python3.12/site-packages/flask_cors/extension.py", line 206, in wrapped_function
    return cors_after_request(app_any.make_response(f(*args, **kwargs)))
                                                    ^^^^^^^^^^^^^^^^^^
  File "/opt/miniconda3/envs/multiagentes/lib/python3.12/site-packages/flask/app.py", line 917, in full_dispatch_request
    rv = self.dispatch_request()
         ^^^^^^